<a href="https://colab.research.google.com/github/praveenkumar-2603/ITA0513-computer-vision/blob/main/assissment.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [3]:
import cv2
import time
from ultralytics import YOLO


# ============================================================
#  REAL-TIME VEHICLE DETECTION AND TRACKING
#  OpenCV + YOLO Hybrid Computer Vision Pipeline
# ============================================================

# -----------------------------
# 1. CONFIGURATION
# -----------------------------

VIDEO_SOURCE = "traffic.mp4"
MODEL_PATH = "yolo11n.pt"

CONFIDENCE_THRESHOLD = 0.50
FRAME_WIDTH = 640
FRAME_HEIGHT = 480

# Vehicle classes available in COCO-trained YOLO models
VEHICLE_CLASSES = {
    "car",
    "motorcycle",
    "bus",
    "truck"
}


# -----------------------------
# 2. LOAD YOLO MODEL
# -----------------------------

print("Loading YOLO model...")

model = YOLO(MODEL_PATH)

print("YOLO model loaded successfully.")


# -----------------------------
# 3. OPEN VIDEO
# -----------------------------

cap = cv2.VideoCapture(VIDEO_SOURCE)

if not cap.isOpened():
    print("ERROR: Cannot open video.")
    print("Check the video path:", VIDEO_SOURCE)
    exit()


# -----------------------------
# 4. BACKGROUND SUBTRACTION
# -----------------------------

background_subtractor = cv2.createBackgroundSubtractorMOG2(
    history=500,
    varThreshold=50,
    detectShadows=True
)


# -----------------------------
# 5. MORPHOLOGICAL KERNEL
# -----------------------------

kernel = cv2.getStructuringElement(
    cv2.MORPH_ELLIPSE,
    (5, 5)
)


# -----------------------------
# 6. PERFORMANCE VARIABLES
# -----------------------------

total_frames = 0
total_vehicles = 0

previous_time = time.time()


# ============================================================
# MAIN PROCESSING LOOP
# ============================================================

while True:

    # --------------------------------------------------------
    # 7. READ FRAME
    # --------------------------------------------------------

    ret, frame = cap.read()

    if not ret:
        print("Video completed.")
        break

    total_frames += 1

    start_time = time.time()


    # --------------------------------------------------------
    # 8. RESIZE FRAME
    # --------------------------------------------------------

    frame = cv2.resize(
        frame,
        (FRAME_WIDTH, FRAME_HEIGHT)
    )


    # --------------------------------------------------------
    # 9. PREPROCESSING
    # --------------------------------------------------------

    # Gaussian filtering for noise reduction
    blurred = cv2.GaussianBlur(
        frame,
        (5, 5),
        0
    )


    # --------------------------------------------------------
    # 10. GRAYSCALE CONVERSION
    # --------------------------------------------------------

    gray = cv2.cvtColor(
        blurred,
        cv2.COLOR_BGR2GRAY
    )


    # --------------------------------------------------------
    # 11. CONTRAST ENHANCEMENT
    # --------------------------------------------------------

    clahe = cv2.createCLAHE(
        clipLimit=2.0,
        tileGridSize=(8, 8)
    )

    enhanced = clahe.apply(gray)


    # --------------------------------------------------------
    # 12. EDGE DETECTION
    # --------------------------------------------------------

    edges = cv2.Canny(
        enhanced,
        50,
        150
    )


    # --------------------------------------------------------
    # 13. MOTION SEGMENTATION
    # --------------------------------------------------------

    motion_mask = background_subtractor.apply(
        blurred
    )


    # --------------------------------------------------------
    # 14. MORPHOLOGICAL PROCESSING
    # --------------------------------------------------------

    motion_mask = cv2.morphologyEx(
        motion_mask,
        cv2.MORPH_OPEN,
        kernel
    )

    motion_mask = cv2.morphologyEx(
        motion_mask,
        cv2.MORPH_CLOSE,
        kernel
    )


    # Remove small regions
    _, motion_mask = cv2.threshold(
        motion_mask,
        200,
        255,
        cv2.THRESH_BINARY
    )


    # --------------------------------------------------------
    # 15. CONTOUR / FEATURE EXTRACTION
    # --------------------------------------------------------

    contours, _ = cv2.findContours(
        motion_mask,
        cv2.RETR_EXTERNAL,
        cv2.CHAIN_APPROX_SIMPLE
    )

    motion_regions = []

    for contour in contours:

        area = cv2.contourArea(contour)

        if area > 500:

            x, y, w, h = cv2.boundingRect(
                contour
            )

            motion_regions.append(
                (x, y, w, h)
            )


    # --------------------------------------------------------
    # 16. YOLO OBJECT DETECTION
    # --------------------------------------------------------

    results = model(
        frame,
        conf=CONFIDENCE_THRESHOLD,
        verbose=False
    )


    # Current frame vehicle count
    frame_vehicle_count = 0


    # --------------------------------------------------------
    # 17. PROCESS YOLO DETECTIONS
    # --------------------------------------------------------

    for result in results:

        if result.boxes is None:
            continue


        for box in result.boxes:

            # Class ID
            class_id = int(
                box.cls[0]
            )

            # Confidence
            confidence = float(
                box.conf[0]
            )

            # Class name
            class_name = model.names[
                class_id
            ]


            # ------------------------------------------------
            # 18. FILTER VEHICLES
            # ------------------------------------------------

            if class_name not in VEHICLE_CLASSES:
                continue


            frame_vehicle_count += 1


            # ------------------------------------------------
            # 19. GET BOUNDING BOX
            # ------------------------------------------------

            x1, y1, x2, y2 = map(
                int,
                box.xyxy[0]
            )


            # ------------------------------------------------
            # 20. DRAW BOUNDING BOX
            # ------------------------------------------------

            cv2.rectangle(
                frame,
                (x1, y1),
                (x2, y2),
                (0, 255, 0),
                2
            )


            # ------------------------------------------------
            # 21. DISPLAY LABEL
            # ------------------------------------------------

            label = (
                f"{class_name} "
                f"{confidence * 100:.1f}%"
            )

            cv2.rectangle(
                frame,
                (x1, y1 - 30),
                (x1 + 180, y1),
                (0, 255, 0),
                -1
            )

            cv2.putText(
                frame,
                label,
                (x1 + 5, y1 - 8),
                cv2.FONT_HERSHEY_SIMPLEX,
                0.5,
                (0, 0, 0),
                1
            )


            # ------------------------------------------------
            # 22. DRAW CENTER POINT
            # ------------------------------------------------

            center_x = int(
                (x1 + x2) / 2
            )

            center_y = int(
                (y1 + y2) / 2
            )

            cv2.circle(
                frame,
                (center_x, center_y),
                4,
                (255, 0, 0),
                -1
            )


    # --------------------------------------------------------
    # 23. TOTAL VEHICLE COUNT
    # --------------------------------------------------------

    total_vehicles += frame_vehicle_count


    # --------------------------------------------------------
    # 24. FPS CALCULATION
    # --------------------------------------------------------

    processing_time = (
        time.time() - start_time
    )

    if processing_time > 0:
        fps = 1.0 / processing_time
    else:
        fps = 0


    # --------------------------------------------------------
    # 25. LATENCY CALCULATION
    # --------------------------------------------------------

    latency_ms = (
        processing_time * 1000
    )


    # --------------------------------------------------------
    # 26. DISPLAY INFORMATION
    # --------------------------------------------------------

    cv2.rectangle(
        frame,
        (0, 0),
        (300, 105),
        (0, 0, 0),
        -1
    )


    cv2.putText(
        frame,
        f"Vehicles: {frame_vehicle_count}",
        (10, 25),
        cv2.FONT_HERSHEY_SIMPLEX,
        0.65,
        (255, 255, 255),
        2
    )


    cv2.putText(
        frame,
        f"FPS: {fps:.2f}",
        (10, 50),
        cv2.FONT_HERSHEY_SIMPLEX,
        0.65,
        (255, 255, 255),
        2
    )


    cv2.putText(
        frame,
        f"Latency: {latency_ms:.1f} ms",
        (10, 75),
        cv2.FONT_HERSHEY_SIMPLEX,
        0.60,
        (255, 255, 255),
        2
    )


    cv2.putText(
        frame,
        f"Frame: {total_frames}",
        (10, 100),
        cv2.FONT_HERSHEY_SIMPLEX,
        0.60,
        (255, 255, 255),
        2
    )


    # --------------------------------------------------------
    # 27. DISPLAY OUTPUT
    # --------------------------------------------------------

    cv2.imshow(
        "OpenCV + YOLO Vehicle Detection",
        frame
    )


    # --------------------------------------------------------
    # 28. OPTIONAL DEBUG WINDOWS
    # --------------------------------------------------------

    # Press M to see motion segmentation
    key = cv2.waitKey(1) & 0xFF

    if key == ord("m"):

        cv2.imshow(
            "Motion Segmentation",
            motion_mask
        )

        cv2.imshow(
            "Edge Detection",
            edges
        )


    # --------------------------------------------------------
    # 29. EXIT
    # --------------------------------------------------------

    if key == ord("q"):
        break


# ============================================================
# 30. RELEASE RESOURCES
# ============================================================

cap.release()

cv2.destroyAllWindows()


# ============================================================
# 31. FINAL RESULTS
# ============================================================

print("\n==========================================")
print("       FINAL SYSTEM RESULTS")
print("==========================================")

print(
    "Total Frames Processed:",
    total_frames
)

print(
    "Total Vehicle Detections:",
    total_vehicles
)

if total_frames > 0:

    average_detections = (
        total_vehicles / total_frames
    )

    print(
        "Average Vehicles / Frame:",
        f"{average_detections:.2f}"
    )

print("System execution completed.")
print("==========================================")

Creating new Ultralytics Settings v0.0.8 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart#ultralytics-settings.
Loading YOLO model...
YOLO model loaded successfully.
ERROR: Cannot open video.
Check the video path: traffic.mp4
Video completed.

       FINAL SYSTEM RESULTS
Total Frames Processed: 0
Total Vehicle Detections: 0
System execution completed.


In [2]:
pip install opencv-python ultralytics numpy

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 46.0/46.0 kB 2.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 36.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 53.2/53.2 kB 3.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 66.2/66.2 kB 4.5 MB/s eta 0:00:00
